## Production Ready RAG with vLLM 

In [ ]:
# Rag Pipeline


import logging
import warnings
from pathlib import Path
from typing import List

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

# Suppress vectorstore deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain_community")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Constants
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
INDEX_PATH = "faiss_docling_index"
DEFAULT_FILE_PATH = Path("./data/paper.pdf")


def process_and_chunk_document(file_path: Path, model_name: str = MODEL_NAME) -> List[Document]:
    """Converts a document using Docling and chunks it into LangChain Documents."""
    logger.info("Initializing converter and tokenizer...")
    tokenizer = HuggingFaceTokenizer(
        tokenizer=AutoTokenizer.from_pretrained(model_name)
    )

    doc_converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.DOCX,
            InputFormat.PPTX,
            InputFormat.ASCIIDOC,
            InputFormat.MD,
            InputFormat.XLSX,
        ],
    )

    logger.info("Processing document: %s", file_path)
    conv_result = doc_converter.convert(file_path)
    source_name = conv_result.input.file.name
    docling_document = conv_result.document

    # Chunk document into LangChain Document format
    chunker = HybridChunker(tokenizer=tokenizer, max_tokens=256, overlap=50)
    documents: List[Document] = []

    for doc_id, chunk in enumerate(chunker.chunk(docling_document), start=1):
        refs = " ".join(item.get_ref().cref for item in chunk.meta.doc_items)
        
        documents.append(
            Document(
                page_content=chunk.text,
                metadata={
                    "doc_id": doc_id,
                    "source": source_name,
                    "ref": refs,
                },
            )
        )

    logger.info("Created %d chunks.", len(documents))
    return documents


def build_and_save_vectorstore(
    documents: List[Document], 
    embedding_model: HuggingFaceEmbeddings, 
    save_path: str = INDEX_PATH
) -> FAISS:
    """Creates a FAISS vectorstore from documents and saves it locally."""
    logger.info("Building FAISS vectorstore...")
    vectorstore = FAISS.from_documents(documents, embedding_model)
    vectorstore.save_local(save_path)
    logger.info("Vectorstore saved to '%s'", save_path)
    return vectorstore


def load_retriever(
    embedding_model: HuggingFaceEmbeddings, 
    load_path: str = INDEX_PATH, 
    top_k: int = 5
):
    """Loads a saved FAISS index and returns a retriever instance."""
    logger.info("Loading vectorstore from '%s'...", load_path)
    vectorstore = FAISS.load_local(
        load_path,
        embedding_model,
        allow_dangerous_deserialization=True,
    )
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k}
    )


# def main():
#     # Initialize shared embedding model
#     embedding_model = HuggingFaceEmbeddings(model_name=MODEL_NAME)

#     # 1. Parse and chunk document
#     documents = process_and_chunk_document(DEFAULT_FILE_PATH)

#     # 2. Build and persist index
#     build_and_save_vectorstore(documents, embedding_model)

#     # 3. Load retriever ready for querying
#     retriever = load_retriever(embedding_model)
    
#     return retriever


# if __name__ == "__main__":
#     retriever = main()


/teamspace/studios/this_studio/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_51537/1721219564.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP

In [3]:

# Initialize shared embedding model
embedding_model = HuggingFaceEmbeddings(model_name=MODEL_NAME)

# 1. Parse and chunk document
documents = process_and_chunk_document(DEFAULT_FILE_PATH)

# 2. Build and persist index
build_and_save_vectorstore(documents, embedding_model)

# 3. Load retriever ready for querying
retriever = load_retriever(embedding_model)


INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6

In [41]:
# vllm serve Qwen/Qwen3-0.6B --dtype=bfloat16 --max-model-len 4096
# vllm serve Qwen/Qwen3-0.6B --dtype float16
# vllm serve Qwen/Qwen3-0.6B --dtype float16 --gpu-memory-utilization 0.80



# vllm serve Qwen3-0.6B-W4A16 --dtype float16 --gpu-memory-utilization 0.80



import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, math, sys

VLLM_URL = "http://localhost:8000"
os.makedirs("outputs", exist_ok=True)
VLLM_USE_FLASHINFER_SAMPLER = 0 

print("Waiting for vLLM server...")
for attempt in range(60):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            MODEL = r.json()["data"][0]["id"]
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
    if attempt % 6 == 5:
        print(f"  Still waiting... ({(attempt + 1) * 5}s elapsed)")
else:
    raise RuntimeError(
        "vLLM server not reachable after 5 minutes."
    )

print(f"Connected to {VLLM_URL} — model: {MODEL}")

Waiting for vLLM server...
Connected to http://localhost:8000 — model: ./models/Qwen3-0.6B-W4A16


In [45]:
from openai import OpenAI   #type: ignore

client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")

In [46]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20, 
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {resp.usage.total_tokens} total")

INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


Response (1.26s, 21 tokens):
PagedAttention is a type of attention mechanism used in language models to optimize parameter sharing between layers.

Usage: 21 prompt + 21 completion = 42 total


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# model_name = 'Qwen/Qwen3-0.6B'
model_name = 'Qwen3-0.6B-W4A16'

# Initialize the vLLM-backed chat model with thinking mode disabled
llm = ChatOpenAI(
    model=model_name,
    base_url=f"{VLLM_URL}/v1",
    api_key="unused",
    temperature=0,
    model_kwargs={
        "extra_body": {
            "chat_template_kwargs": {       # disable Thinking mode
                "enable_thinking": False
            }
        }
    }
)

prompt = PromptTemplate.from_template(
    """
You are a highly reliable technical assistant.  
Answer ONLY using the information provided in the context.  
If the answer is not explicitly stated in the context, respond with:
"I don't have enough information from the provided documents to answer this."

STRICT RULES:
- Do NOT use prior knowledge.
- Do NOT guess or hallucinate.
- Do NOT add missing technical details.
- Do NOT assume beyond what the context supports.
- Only answer if the context contains clear and direct evidence.

When answering:
- Use clear, concise technical language.
- If multiple pieces of context conflict, point out the conflict instead of guessing.
- If the context includes irrelevant information, ignore it.

--------------------
CONTEXT:
{context}
--------------------

QUESTION:
{question}

FINAL ANSWER (based ONLY on context):
"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

query = "What are the three distinct RAG paradigms defined in the survey paper, and how do their workflows differ?"

response = qa_chain.invoke({"query": query})

print(response["result"])

INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


The three distinct RAG paradigms defined in the survey paper are Naive RAG, Advanced RAG, and Modular RAG. The workflows differ as follows: Naive RAG relies on a simple retrieval process to gather information from external sources, while Advanced RAG integrates more sophisticated techniques to handle complex tasks and data. Modular RAG allows for flexible integration of different components to address specific challenges.


In [47]:
def get_vllm_metrics(base_url=VLLM_URL):
    """Scrape vLLM Prometheus /metrics and return {name: value}."""
    r = requests.get(f"{base_url}/metrics")
    metrics = {}
    for line in r.text.split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except (ValueError, IndexError):
            continue
    return metrics

metrics = get_vllm_metrics()
print("Current vLLM Metrics:")
for key in ["vllm:num_requests_running", "vllm:num_requests_waiting",
            "vllm:gpu_cache_usage_perc", "vllm:cpu_cache_usage_perc",
            "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]:
    if key in metrics:
        print(f"  {key.replace('vllm:', '')}: {metrics[key]:g}")

with open("outputs/metrics_snapshot.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nFull metrics saved to outputs/metrics_snapshot.json")

Current vLLM Metrics:
  num_requests_running: 0
  num_requests_waiting: 0
  prompt_tokens_total: 21
  generation_tokens_total: 21

Full metrics saved to outputs/metrics_snapshot.json


In [ ]:
import time
import concurrent.futures
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


# model_name = 'Qwen/Qwen3-0.6B'
model_name = 'Qwen3-0.6B-W4A16'

# llm = ChatOpenAI(
#     model=model_name,
#     base_url=f"{VLLM_URL}/v1",
#     api_key="unused",
#     temperature=0,
# )


llm = ChatOpenAI(
    model=model_name,
    base_url=f"{VLLM_URL}/v1",
    api_key="unused",
    temperature=0,
    model_kwargs={
        "extra_body": {
            "chat_template_kwargs": {       # disable Thinking mode
                "enable_thinking": False
            }
        }
    }
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a highly reliable technical assistant.
Answer ONLY using the information provided in the context.
If the answer is not explicitly stated in the context, respond with:
"I don't have enough information from the provided documents to answer this."

STRICT RULES:
- Do NOT use prior knowledge.
- Do NOT guess or hallucinate.
- Do NOT add missing technical details.
- Do NOT assume beyond what the context supports.
- Only answer if the context contains clear and direct evidence.

--------------------
CONTEXT:
{context}
--------------------

QUESTION:
{question}

FINAL ANSWER (based ONLY on context):
"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

queries = [
    "What are the three distinct RAG paradigms defined in the survey paper, and how do their workflows differ?",
    "What is the difference between naive RAG and advanced RAG?",
    "How does retrieval augmentation help reduce hallucination?",
    "What evaluation metrics are used for RAG systems in the survey?",
    "What are the main components of a modular RAG architecture?",
]

def _ask(query):
    return qa_chain.invoke({"query": query})

before = get_vllm_metrics()
print(f"Sending {len(queries)} concurrent RAG queries...\n")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=len(queries)) as pool:
    futures = {pool.submit(_ask, q): q for q in queries}

    time.sleep(0.5)
    during = get_vllm_metrics()
    running = during.get("vllm:num_requests_running", "--")
    waiting = during.get("vllm:num_requests_waiting", "--")
    print(f"  [mid-flight]  running: {running}  |  waiting: {waiting}")

    results = {}
    for f in concurrent.futures.as_completed(futures):
        q = futures[f]
        resp = f.result()
        results[q] = resp["result"]
        print(f"  done: \"{q[:50]}...\"")

elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0)

print(f"\nAll {len(queries)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

for q, answer in results.items():
    print(f"\nQ: {q}\nA: {answer}")

Sending 5 concurrent RAG queries...

  [mid-flight]  running: 4.0  |  waiting: 1.0


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "How does retrieval augmentation help reduce halluc..."
  done: "What evaluation metrics are used for RAG systems i..."
  done: "What are the main components of a modular RAG arch..."


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "What is the difference between naive RAG and advan..."


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "What are the three distinct RAG paradigms defined ..."

All 5 completed in 2.21s
Tokens generated: 181  |  ~81.9 tokens/s

Q: How does retrieval augmentation help reduce hallucination?
A: I don't have enough information from the provided documents to answer this.

Q: What evaluation metrics are used for RAG systems in the survey?
A: BLEU, Faithfulness = ✓. BLEU, Answer Relevance = ✓.

Q: What are the main components of a modular RAG architecture?
A: The main components of a modular RAG architecture are the retrieval module, the generation module, and the augmentation module.

Q: What is the difference between naive RAG and advanced RAG?
A: The difference between naive RAG and advanced RAG is that naive RAG focuses on improving retrieval quality and addressing indexing issues through pre-retrieval and post-retrieval strategies, while advanced RAG incorporates additional optimization methods to streamline the retrieval process.

Q: What are the three distinct RAG paradigms defin

In [ ]:
import time
import concurrent.futures
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


model_name = 'Qwen/Qwen3-0.6B'
# model_name = './models/Qwen3-0.6B-W4A16'

# llm = ChatOpenAI(
#     model=model_name,
#     base_url=f"{VLLM_URL}/v1",
#     api_key="unused",
#     temperature=0,
# )


llm = ChatOpenAI(
    model=model_name,
    base_url=f"{VLLM_URL}/v1",
    api_key="unused",
    temperature=0,
    model_kwargs={
        "extra_body": {
            "chat_template_kwargs": {       # disable Thinking mode
                "enable_thinking": False
            }
        }
    }
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a highly reliable technical assistant.
Answer ONLY using the information provided in the context.
If the answer is not explicitly stated in the context, respond with:
"I don't have enough information from the provided documents to answer this."

STRICT RULES:
- Do NOT use prior knowledge.
- Do NOT guess or hallucinate.
- Do NOT add missing technical details.
- Do NOT assume beyond what the context supports.
- Only answer if the context contains clear and direct evidence.

--------------------
CONTEXT:
{context}
--------------------

QUESTION:
{question}

FINAL ANSWER (based ONLY on context):
"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

queries = [
    "What are the three distinct RAG paradigms defined in the survey paper, and how do their workflows differ?",
    "What is the difference between naive RAG and advanced RAG?",
    "How does retrieval augmentation help reduce hallucination?",
    "What evaluation metrics are used for RAG systems in the survey?",
    "What are the main components of a modular RAG architecture?",
]

def _ask(query):
    return qa_chain.invoke({"query": query})

before = get_vllm_metrics()
print(f"Sending {len(queries)} concurrent RAG queries...\n")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=len(queries)) as pool:
    futures = {pool.submit(_ask, q): q for q in queries}

    time.sleep(0.5)
    during = get_vllm_metrics()
    running = during.get("vllm:num_requests_running", "--")
    waiting = during.get("vllm:num_requests_waiting", "--")
    print(f"  [mid-flight]  running: {running}  |  waiting: {waiting}")

    results = {}
    for f in concurrent.futures.as_completed(futures):
        q = futures[f]
        resp = f.result()
        results[q] = resp["result"]
        print(f"  done: \"{q[:50]}...\"")

elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0)

print(f"\nAll {len(queries)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

for q, answer in results.items():
    print(f"\nQ: {q}\nA: {answer}")

Sending 5 concurrent RAG queries...

  [mid-flight]  running: 0.0  |  waiting: 0.0


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "How does retrieval augmentation help reduce halluc..."
  done: "What are the main components of a modular RAG arch..."


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "What evaluation metrics are used for RAG systems i..."


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "What are the three distinct RAG paradigms defined ..."


INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"


  done: "What is the difference between naive RAG and advan..."

All 5 completed in 4.43s
Tokens generated: 174  |  ~39.3 tokens/s

Q: How does retrieval augmentation help reduce hallucination?
A: None of the provided information allows for a direct answer about how retrieval augmentation helps reduce hallucination.

Q: What are the main components of a modular RAG architecture?
A: The main components of a modular RAG architecture include structured RAG modules and rearranged RAG pipelines.

Q: What evaluation metrics are used for RAG systems in the survey?
A: The specific metrics used for RAG systems in the survey include BLEU, ROUGE/ROUGE-L, and Context Relevance.

Q: What are the three distinct RAG paradigms defined in the survey paper, and how do their workflows differ?
A: The three distinct RAG paradigms defined in the survey paper are Naive RAG, Advanced RAG, and Modular RAG. The workflows differ significantly, as outlined in the context provided.

Q: What is the difference betwe

In [52]:
for q, answer in results.items():
    print(f"\nQ: {q}\nA: {answer}")


Q: How does retrieval augmentation help reduce hallucination?
A: None of the provided information allows for a direct answer about how retrieval augmentation helps reduce hallucination.

Q: What are the main components of a modular RAG architecture?
A: The main components of a modular RAG architecture include structured RAG modules and rearranged RAG pipelines.

Q: What evaluation metrics are used for RAG systems in the survey?
A: The specific metrics used for RAG systems in the survey include BLEU, ROUGE/ROUGE-L, and Context Relevance.

Q: What are the three distinct RAG paradigms defined in the survey paper, and how do their workflows differ?
A: The three distinct RAG paradigms defined in the survey paper are Naive RAG, Advanced RAG, and Modular RAG. The workflows differ significantly, as outlined in the context provided.

Q: What is the difference between naive RAG and advanced RAG?
A: Naive RAG and Advanced RAG differ primarily in their approach and optimization strategies. Naive 

In [53]:
# import time

# # Print results
# for q, answer in results.items():
#     print(f"\nQ: {q}\nA: {answer}")

# # Keep-alive loop: Sleep for 1 hour, printing every 60 seconds
# KEEP_ALIVE_MINUTES = 60
# print(f"\n[Keep-Alive] Keeping Lightning Studio awake for {KEEP_ALIVE_MINUTES} minutes...")

# for minute in range(1, KEEP_ALIVE_MINUTES + 1):
#     time.sleep(60)  # Sleep for 60 seconds
#     print(f"[Keep-Alive] Heartbeat active... ({minute}/{KEEP_ALIVE_MINUTES} min)")

# print("[Keep-Alive] Done.")

In [44]:
# vllm serve Qwen/Qwen3-0.6B \
#   --dtype float16 \
#   --gpu-memory-utilization 0.80 \
#   --enable-prefix-caching